In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import BaggingRegressor,RandomForestRegressor,StackingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, ElasticNet,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import r2_score, mean_squared_error
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline
import warnings
from tqdm import tqdm
import os
os.chdir("/home/pgcp-ai/MachineLearning/Datasets/Calorie/")

In [4]:
calorie = pd.read_csv("train.csv", index_col = 0)
calorie

,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
id,,,,,,,,
0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,female,38,166.0,61.0,25.0,102.0,40.6,146.0
...,...,...,...,...,...,...,...,...
749995,male,28,193.0,97.0,30.0,114.0,40.9,230.0
749996,female,64,165.0,63.0,18.0,92.0,40.5,96.0
749997,male,60,162.0,67.0,29.0,113.0,40.9,221.0


In [8]:
calorie.isna().sum().sum()

0

In [21]:
X, y = calorie.drop(['Calories','Sex'], axis = 1), calorie['Calories']

In [22]:
cbm = CatBoostRegressor()

In [24]:
folds = KFold(n_splits = 5, shuffle = True, random_state = 26)

params = {"n_estimators" : [2,5,10,20,50]}
gcv = GridSearchCV(estimator = cbm, cv = folds,param_grid = params, n_jobs = -1)
gcv.fit(X, y)

Learning rate set to 0.5
0:	learn: 34.1247213	total: 92.7ms	remaining: 4.54s
1:	learn: 19.8057288	total: 115ms	remaining: 2.76s
2:	learn: 12.7805052	total: 135ms	remaining: 2.11s
3:	learn: 9.4225356	total: 159ms	remaining: 1.82s
4:	learn: 8.0326192	total: 181ms	remaining: 1.63s
5:	learn: 7.4043533	total: 200ms	remaining: 1.46s
6:	learn: 7.0200708	total: 219ms	remaining: 1.35s
7:	learn: 6.6352357	total: 239ms	remaining: 1.25s
8:	learn: 6.4205480	total: 258ms	remaining: 1.17s
9:	learn: 6.2768441	total: 278ms	remaining: 1.11s
10:	learn: 6.0142692	total: 301ms	remaining: 1.06s
11:	learn: 5.9025817	total: 321ms	remaining: 1.02s
12:	learn: 5.7419297	total: 339ms	remaining: 966ms
13:	learn: 5.6134846	total: 360ms	remaining: 925ms
14:	learn: 5.4150220	total: 389ms	remaining: 908ms
15:	learn: 5.3539196	total: 408ms	remaining: 867ms
16:	learn: 5.2644904	total: 427ms	remaining: 828ms
17:	learn: 5.2056053	total: 444ms	remaining: 789ms
18:	learn: 5.1202790	total: 465ms	remaining: 759ms
19:	learn: 5

GridSearchCV(cv=KFold(n_splits=5, random_state=26, shuffle=True),
             estimator=CatBoostRegressor(loss_function='RMSE'), n_jobs=-1,
             param_grid={'n_estimators': [2, 5, 10, 20, 50]})

In [25]:
gcv.best_params_,gcv.best_score_

({'n_estimators': 50}, 0.9950208812529138)

In [26]:
tst = pd.read_csv("test.csv", index_col = 0)
tst.drop(['Sex'], axis = 1, inplace = True)
y_pred = gcv.predict(tst)

In [27]:
submit = pd.read_csv("sample_submission.csv")
submit["Calories"] = y_pred

In [28]:
submit.to_csv("kaggle.csv", index = False)